### 11. I denna uppgift kommer vi arbeta med kategorisk data. 
Nominaldata är kategorisk data där kategorierna inte har någon inbördes rangordning.

> Datasetet `mpg` som följer med biblioteket *seaborn* är ett dataset med 398 observationer av bilar. Den beroende variabeln `mpg` står för *miles per gallon* och beskriver bilarnas bränsleeffektivitet.

> Om vi vill träna en regressionsmodell på `mpg`-datasetet behöver vi hantera de två kategoriska variablerna `origin` och `name`. Variabeln `origin` är en kategorisk variabel med tre olika värden: `europe`, `japan` och `usa`. Den lämpar sig bra för *one hot encoding*. Variabeln `name` har 305 unika värden. Skulle vi utföra *one hot encoding* på den skulle vårt dataset få 305 nya dimensioner vilket inte är så lämpligt i detta fall. Dessutom - tror du att namnet på bilen har något att göra med hur långt den kör på en *gallon* bensin? Vi ska därför droppa `name`-kolumnen innan vi börjar träna en modell på datan.

```python
import seaborn as sns

df = sns.load_dataset("mpg")
print(df.head())
```

#### a) Läs in datasetet `mpg` med seaborns `load_dataset()`-funktion (det gjordes i koden ovan).

In [17]:
import pandas as pd 
import numpy as np
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = sns.load_dataset("mpg")

print(df.head())
df.info()

    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  
3          70    usa              amc rebel sst  
4          70    usa                ford torino  
<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacem

#### b) Droppa rader med saknade värden med `dropna()`-metoden.

In [2]:
df = df.dropna()
df.info()

<class 'pandas.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   model_year    392 non-null    int64  
 7   origin        392 non-null    str    
 8   name          392 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 38.4 KB


#### c) Droppa kolumnen `name`.


In [3]:
df = df.drop(columns=["name"])
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa


#### d) Utför en *dummy-variable-encoding* på `origin`-kolumnen med pandas `get_dummies()`-funktion. Ange `drop_first=True` så att vi får 2 nya kolumner istället för 3.


In [4]:
df = pd.get_dummies(df, columns=["origin"], drop_first=True)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_japan,origin_usa
0,18.0,8,307.0,130.0,3504,12.0,70,False,True
1,15.0,8,350.0,165.0,3693,11.5,70,False,True
2,18.0,8,318.0,150.0,3436,11.0,70,False,True
3,16.0,8,304.0,150.0,3433,12.0,70,False,True
4,17.0,8,302.0,140.0,3449,10.5,70,False,True


#### e) Dela upp datasetet i `X` och `y`, med `mpg` som den beroende variabeln `y`.


In [11]:
X = df.drop(columns=["mpg"])
y = df["mpg"]

#### f) Dela upp datasetet i träning- och testset.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### g) Träna en linjär regressionsmodell på träningsdatan och utvärdera den på testdatan.

In [18]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }
    
mpg_model = LinearRegression()
mpg_model.fit(X_train, y_train)

y_pred = mpg_model.predict(X_test)

mpg_results = pd.Series(regression_metrics(y_test, y_pred), name="Test set")
mpg_results

MAE     2.462000
RMSE    3.256114
Name: Test set, dtype: float64